In [ ]:
"""Task 2: Huffman n-gram coder — анализ на синтетических данных."""

import math
import random
from collections import OrderedDict
from typing import Dict
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from codinglab.huffman_ngram import HuffmanNGramCoder

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
random.seed(42)

Наборы:

размер алфавита: 2 -- минимальный классический, 16 -- средний классический (16-ричная система), 26 -- английский алфавит (для текстов)

размер текста: 1000 -- минимальный стандартный (как был в предыдущем задании), 10000 -- средний используемый для экспериментов, 100000 -- максимальный

распределение: равномерное -- самое базовое и простое, экспоненциальное -- небольшое количество символов с большой частотой, остальные -- нечастые, степенное -- приближенное к распределению в языке

таким образом 27 наборов, для них перебираем n от 1 до 5 и по n ∈ {1, 2, 3, 4, 5, 6, 8, 10, 12, 15}

In [ ]:
def uniform_probs(alphabet_size: int) -> OrderedDict[str, float]:
    """Равномерное распределение: все символы равновероятны"""
    symbols = [chr(ord("a") + i) for i in range(alphabet_size)]
    prob = 1.0 / alphabet_size
    return OrderedDict({s: prob for s in symbols})


def exponential_probs(alphabet_size: int, rate: float = 0.3) -> OrderedDict[str, float]:
    """Экспоненциальное распределение: быстрые символы частые, редкие -- экспоненциально реже"""
    symbols = [chr(ord("a") + i) for i in range(alphabet_size)]
    weights = [math.exp(-rate * i) for i in range(alphabet_size)]
    total = sum(weights)
    probs = [w / total for w in weights]
    return OrderedDict(zip(symbols, probs))


def zipf_probs(alphabet_size: int, exponent: float = 1.0) -> OrderedDict[str, float]:
    """Степенное распределение"""
    symbols = [chr(ord("a") + i) for i in range(alphabet_size)]
    weights = [1.0 / ((i + 1) ** exponent) for i in range(alphabet_size)]
    total = sum(weights)
    probs = [w / total for w in weights]
    return OrderedDict(zip(symbols, probs))


def generate_text(probabilities: OrderedDict[str, float], length: int) -> str:
    """Генерация текста с заданным распределением"""
    symbols = list(probabilities.keys())
    weights = list(probabilities.values())
    return "".join(random.choices(symbols, weights=weights, k=length))

In [ ]:
# Параметры эксперимента
CONFIG = {
    "distributions": {
        "uniform": uniform_probs,
        "exponential": exponential_probs,
        "zipf": zipf_probs,
    },
    "alphabet_sizes": [2, 16, 26],
    "text_lengths": [1_000, 10_000, 100_000],
    "n_values": [1, 2, 3, 4, 5],
}

О метриках:
Нам интересны энтропия, средняя длина кода (т.к. они напрямую рассматриваются в проверяемом равенстве), а также избыточность (именно она показывает разницу между оценкой и фактическим значением длины, которые мы рассматриваем)


In [ ]:
def run_experiment(
    dist_name: str, alphabet_size: int, text_length: int, n: int
) -> Dict:
    """
    Запустить один эксперимент и вернуть метрики.

    Returns:
        Dict с параметрами и результатами
    """
    probs_func = CONFIG["distributions"][dist_name]
    probabilities = probs_func(alphabet_size)
    text = generate_text(probabilities, text_length)

    coder = HuffmanNGramCoder(n=n)
    coder.fit(text)

    return {
        "distribution": dist_name,
        "alphabet_size": alphabet_size,
        "text_length": text_length,
        "n": n,
        "unique_ngrams": coder.alphabet_size,
        "bits_per_symbol": coder.expected_code_length_per_symbol,
        "entropy_per_symbol": coder.entropy_per_symbol,
        "efficiency": coder.coding_efficiency,
        "redundancy_per_symbol": coder.redundancy_per_symbol,
    }

In [ ]:
def run_all_experiments(config: Dict) -> pd.DataFrame:
    """Запустить все комбинации параметров."""
    results = []
    len(config["distributions"]) * len(config["alphabet_sizes"]) * len(
        config["text_lengths"]
    ) * len(config["n_values"])
    count = 0

    for dist_name in config["distributions"]:
        for alphabet_size in config["alphabet_sizes"]:
            for text_length in config["text_lengths"]:
                for n in config["n_values"]:
                    result = run_experiment(dist_name, alphabet_size, text_length, n)
                    if result:
                        results.append(result)
                    count += 1

    return pd.DataFrame(results)


results_df = run_all_experiments(CONFIG)
results_df.head()

In [ ]:
def plot_bits_per_symbol(df: pd.DataFrame):
    """График: средняя длина кода (бит/символ) в зависимости от n."""

    plt.figure(figsize=(14, 7))

    for (dist, alph_size, length), group in df.groupby(
        ["distribution", "alphabet_size", "text_length"]
    ):
        label = f"{dist}, |Σ|={alph_size}, N={length:,}"
        plt.plot(
            group["n"],
            group["bits_per_symbol"],
            marker="o",
            label=label,
            linewidth=1.5,
            markersize=4,
        )

    plt.xlabel("Длина блока n", fontsize=12)
    plt.ylabel("Средняя длина кода (бит/символ)", fontsize=12)
    plt.title("Эффективность сжатия: биты на символ", fontsize=14, fontweight="bold")
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_bits_per_symbol(results_df)

In [ ]:
def plot_redundancy(df: pd.DataFrame):
    """Сравнение фактической избыточности с теоретической границей 1/n."""

    # Берём один типичный набор для наглядности
    subset = df[
        (df["distribution"] == "uniform")
        & (df["alphabet_size"] == 16)
        & (df["text_length"] == 10_000)
    ].copy()

    if len(subset) == 0:
        print("⚠️  Нет данных для этой конфигурации")
        return

    plt.figure(figsize=(10, 6))

    # Фактическая избыточность
    plt.plot(
        subset["n"],
        subset["redundancy_per_symbol"],
        marker="o",
        label="Фактическая (L - H)",
        linewidth=2,
        color="blue",
    )

    # Теоретическая граница: 1/n
    theoretical = [1 / n for n in subset["n"]]
    plt.plot(
        subset["n"],
        theoretical,
        marker="x",
        label="Теоретическая граница (1/n)",
        linewidth=2,
        color="red",
        linestyle="--",
    )

    plt.xlabel("Длина блока n", fontsize=12)
    plt.ylabel("Избыточность (бит/символ)", fontsize=12)
    plt.title("Сравнение с теоретической оценкой", fontsize=14, fontweight="bold")
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    display_df = subset[
        ["n", "bits_per_symbol", "entropy_per_symbol", "redundancy_per_symbol"]
    ].copy()
    display_df["theoretical_1/n"] = [1 / n for n in display_df["n"]]
    print(display_df.to_string(index=False))


plot_redundancy(results_df)

In [ ]:
# Расширенная конфигурация для больших n
CONFIG_EXTENDED = {
    "distributions": {
        "uniform": uniform_probs,
    },
    "alphabet_sizes": [2, 4],
    "text_lengths": [10_000, 100_000],
    "n_values": [1, 2, 3, 4, 5, 6, 8, 10, 12, 15],
}


def plot_extended_range(df: pd.DataFrame):
    """График для расширенного диапазона n."""

    subset = df[
        (df["alphabet_size"].isin([2, 4])) & (df["distribution"] == "uniform")
    ].copy()

    plt.figure(figsize=(14, 7))

    for (alph_size, length), group in subset.groupby(["alphabet_size", "text_length"]):
        label = f"|Σ|={alph_size}, N={length:,}"

        # Энтропия H
        plt.plot(
            group["n"],
            group["entropy_per_symbol"],
            marker="o",
            label=f"H ({label})",
            linewidth=2,
            linestyle="-",
        )

        # Средняя длина L/n
        plt.plot(
            group["n"],
            group["bits_per_symbol"],
            marker="s",
            label=f"L/n ({label})",
            linewidth=2,
        )

        # Верхняя граница H + 1/n
        upper_bound = group["entropy_per_symbol"] + 1 / group["n"]
        plt.plot(
            group["n"],
            upper_bound,
            marker="x",
            linestyle="--",
            alpha=0.5,
        )

    plt.xlabel("Длина блока n", fontsize=12)
    plt.ylabel("Биты на символ", fontsize=12)
    plt.title(
        "Неравенство Хаффмана: H ≤ L/n < H + 1/n (расширенный диапазон)",
        fontsize=14,
        fontweight="bold",
    )
    plt.legend(fontsize=9, ncol=2)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


results_extended_df = run_all_experiments(CONFIG_EXTENDED)

plot_extended_range(results_extended_df)

На графиках видим, что нижняя граница (энтропия) начинает совпадать со средней длиной, тогда как верхняя слишком грубо оценивает при небольших n